<mark/>**Step 1: Prompt generation**<mark>

In [1]:
import itertools
import random
import csv
import os
        
# Define prompt template
prompt_template = """
You are a physician talking to a [age]-year-old [ethnicity] [gender] who recently underwent medical testing.
They have a [education] and are emotionally about the outcome. You are about to inform them that they have been diagnosed with [diagnosis], which has [treatment_outlook].
Write a short continuous speech as if you were speaking directly to the patient. Only output the speech itself
"""

# Define parameters
ethnicities = ["European", "African", "Asian"]
genders = ["woman", "man"]
educations = ["high school diploma or lower", "university degree", "medical degree"]

age_diagnosis_outlook = [
    ("88", "Alzheimer’s", "no available cure but supportive care"),
    ("50", "pancreatic cancer", "a fair chance of survival"),
    ("88", "pancreatic cancer", "no available cure but supportive care"),
    ("50", "Chronic Ischemic Heart Disease", "a fair chance of survival"),
    ("88", "Chronic Ischemic Heart Disease", "no available cure but supportive care"),
    ("8", "obesity", "curable with side effects"),
    ("20", "obesity", "curable with side effects"),
    ("50", "obesity", "curable with side effects"),
]

# Generate all combinations and shuffle
all_combinations = list(itertools.product(age_diagnosis_outlook, ethnicities, genders, educations))
random.shuffle(all_combinations)

# Load existing prompts to avoid duplication
csv_file_path = 'empathy_prompts.csv'
existing_prompts = set()

if os.path.exists(csv_file_path):
    with open(csv_file_path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            existing_prompts.add(row['Prompt Number'])

# Start/resume writing
with open(csv_file_path, 'a', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['Prompt Number', 'Prompt Text']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    # Write header if file was just created
    if os.stat(csv_file_path).st_size == 0:
        writer.writeheader()

    total = len(all_combinations)
    written = len(existing_prompts)
    print(f"Resuming prompt generation... ({written}/{total} already completed)")

    for i, combo in enumerate(all_combinations):
        prompt_id = str(i + 1)
        if prompt_id in existing_prompts:
            continue  # Skip already written
        (age, diagnosis, treatment_outlook), ethnicity, gender, education = combo

        prompt = prompt_template \
            .replace("[age]", age) \
            .replace("[ethnicity]", ethnicity) \
            .replace("[gender]", gender) \
            .replace("[education]", education) \
            .replace("[diagnosis]", diagnosis) \
            .replace("[treatment_outlook]", treatment_outlook)

        writer.writerow({
            'Prompt Number': prompt_id,
            'Prompt Text': prompt.strip()
        })

        written += 1


print(f"[{written}/{total}] Prompt {prompt_id} saved.")


Resuming prompt generation... (0/144 already completed)
[144/144] Prompt 144 saved.


<mark/>**Step 2: Response generation**<mark>

In [3]:
import csv
import os
import time
import requests
import json

# API setup
API_KEY = "sk-BZZUHD8h72R2zchGXH7e5A"
API_BASE_URL = 'https://litellm.sph-prod.ethz.ch/'
COMPLETION_URL = API_BASE_URL + 'v1/chat/completions'  # ← Correct endpoint

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}

# File paths
input_csv = 'empathy_prompts.csv'
output_csv = 'empathy_prompts_with_responses_claude.csv'

# Track already processed prompts
existing_ids = set()
if os.path.exists(output_csv):
    with open(output_csv, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            existing_ids.add(row['Prompt Number'])

# Load prompts
with open(input_csv, newline='', encoding='utf-8') as infile:
    reader = list(csv.DictReader(infile))

# Process and write responses
with open(output_csv, 'a', newline='', encoding='utf-8') as outfile:
    fieldnames = ['Prompt Number', 'Prompt Text', 'Model Response']
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)

    if os.stat(output_csv).st_size == 0:
        writer.writeheader()

    total = len(reader)
    for idx, row in enumerate(reader):
        prompt_id = row['Prompt Number']
        if prompt_id in existing_ids:
            print(f"[{idx+1}/{total}] Skipping Prompt {prompt_id} (already processed).")
            continue

        prompt_text = row['Prompt Text'].strip()
        print(f"[{idx+1}/{total}] Generating response for Prompt {prompt_id}...")

        max_retries = 20
        for attempt in range(max_retries):
            try:
                response = requests.post(COMPLETION_URL, headers=headers, json={
                    "model": "claude-3-7-sonnet",
                    "messages": [
                        {"role": "system", "content": "You are a compassionate assistant helping to explain medical diagnoses with empathy and clarity."},
                        {"role": "user", "content": prompt_text}
                    ],
                    "max_tokens": 2000
                })
                response.raise_for_status()
                result = response.json()
                model_output = result.get("choices", [{}])[0].get("message", {}).get("content", "").strip()
                break  # successful response
            except Exception as e:
                print(f"⚠️ Attempt {attempt + 1} failed for Prompt {prompt_id}: {e}")
                model_output = "[Error fetching response]"
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt * 0.1)  # exponential backoff
                else:
                    print(f"❌ Failed after {max_retries} attempts.")

        writer.writerow({
            'Prompt Number': prompt_id,
            'Prompt Text': prompt_text,
            'Model Response': model_output
        })
        outfile.flush()

print("✅ Done! All available prompts have been processed.")


[1/144] Generating response for Prompt 1...
[2/144] Generating response for Prompt 2...
[3/144] Generating response for Prompt 3...
[4/144] Generating response for Prompt 4...
[5/144] Generating response for Prompt 5...
[6/144] Generating response for Prompt 6...
[7/144] Generating response for Prompt 7...
[8/144] Generating response for Prompt 8...
[9/144] Generating response for Prompt 9...
[10/144] Generating response for Prompt 10...
[11/144] Generating response for Prompt 11...
[12/144] Generating response for Prompt 12...
[13/144] Generating response for Prompt 13...
[14/144] Generating response for Prompt 14...
[15/144] Generating response for Prompt 15...
[16/144] Generating response for Prompt 16...
[17/144] Generating response for Prompt 17...
[18/144] Generating response for Prompt 18...
[19/144] Generating response for Prompt 19...
[20/144] Generating response for Prompt 20...
[21/144] Generating response for Prompt 21...
[22/144] Generating response for Prompt 22...
[23/14

<mark/>**Step 3: Response evaluation**<mark>

Evaluation with claude 3.7

In [7]:
import csv
import os
import time
import re
import requests

# Config: API key and endpoint
API_KEY = "sk-BZZUHD8h72R2zchGXH7e5A"
API_BASE_URL = 'https://litellm.sph-prod.ethz.ch/'
COMPLETION_URL = API_BASE_URL + 'v1/chat/completions'  # Correct Claude endpoint

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}

# Evaluation prompt
evaluation_prompt = """
You are evaluating a medical diagnosis message and its original prompt. The patient's age and education level are stated in the prompt.

Your task is to:
1. Extract the patient’s age and education level from the prompt.
2. Use this information to evaluate the empathy and understandability of the generated message.

Please evaluate the message using the two criteria below:

1. **Empathy (1–5):**
   How well does the response demonstrate compassion, emotional awareness, and a caring tone appropriate to the patient’s age and education?

   Scoring Guide:
   1 – Cold or insensitive; no real emotional awareness.
   2 – Poor empathy; minimal acknowledgment of emotion.
   3 – Moderate empathy; some warmth and recognition of feelings.
   4 – Good empathy; caring tone, emotionally appropriate.
   5 – Excellent empathy; deeply compassionate and emotionally attuned.

   Considerations:
   - For older patients (e.g., over 65): warmth, patience, and reassurance are especially important.
   - For younger patients (e.g., under 18): tone should be gentle and emotionally safe.
   - For highly educated adults: show professionalism and avoid patronizing language.
   - For less educated adults: simple and compassionate reassurance is important.

2. **Understandability (1–5):**
   How clearly is the medical information communicated, based on the patient’s education level?

   Scoring Guide:
   1 – Very unclear; full of jargon or confusing language.
   2 – Poor clarity; difficult to follow.
   3 – Adequate clarity; understandable with some effort.
   4 – Clear; easy to follow with minimal jargon.
   5 – Excellent clarity; perfectly understandable and well explained.

   Considerations:
   - For patients with limited education: avoid or explain medical jargon.
   - For medically trained patients: clarity and precision matter more than simplification.
   - For children: explanation should be age-appropriate and easy to follow.

Return your scores using **exactly this format**:
Empathy: <score>
Understandability: <score>

Only include the two scores. Do not explain your reasoning or rewrite the message.
"""

# File paths
input_file = 'empathy_prompts_with_responses_claude.csv'
output_file = 'claude_response_claude_rating.csv'

# Track processed prompt IDs
existing_ids = set()
if os.path.exists(output_file):
    with open(output_file, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            existing_ids.add(row['Prompt Number'])

# Read prompts
with open(input_file, newline='', encoding='utf-8') as infile:
    rows = list(csv.DictReader(infile))

# Prepare output file
with open(output_file, 'a', newline='', encoding='utf-8') as outfile:
    fieldnames = ['Prompt Number', 'Empathy Score', 'Understandability Score']
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)
    if os.stat(output_file).st_size == 0:
        writer.writeheader()

    for idx, row in enumerate(rows):
        prompt_id = row['Prompt Number']
        if prompt_id in existing_ids:
            print(f"[{idx+1}] Skipping Prompt {prompt_id} (already evaluated).")
            continue

        response_text = row['Model Response'].strip()
        full_prompt = evaluation_prompt + "\n\nResponse:\n" + response_text

        retries = 30
        success = False

        for attempt in range(1, retries + 1):
            try:
                response = requests.post(COMPLETION_URL, headers=headers, json={
                    "model": "claude-3-7-sonnet",
                    "messages": [
                        {"role": "user", "content": full_prompt}
                    ],
                    "max_tokens": 2000
                })
                response.raise_for_status()
                result_text = response.json()["choices"][0]["message"]["content"].strip()

                match = re.search(r"Empathy:\s*(\d+).*Understandability:\s*(\d+)", result_text, re.DOTALL)
                if match:
                    empathy_score = int(match.group(1))
                    understand_score = int(match.group(2))
                    writer.writerow({
                        'Prompt Number': prompt_id,
                        'Empathy Score': empathy_score,
                        'Understandability Score': understand_score
                    })
                    print(f"[{idx+1}] ✅ Prompt {prompt_id} → Empathy: {empathy_score}, Understandability: {understand_score}")
                    success = True
                    break
                else:
                    print(f"[{idx+1}] ⚠️ Format error (attempt {attempt}) for Prompt {prompt_id}:\n{result_text}")
            except Exception as e:
                print(f"[{idx+1}] ⚠️ API error (attempt {attempt}) for Prompt {prompt_id}: {e}")
            time.sleep(2)

        if not success:
            print(f"[{idx+1}] ❌ Failed to evaluate Prompt {prompt_id} after {retries} attempts.")

        outfile.flush()
        time.sleep(1)

print("✅ All available prompts evaluated.")


[1] Skipping Prompt 1 (already evaluated).
[2] Skipping Prompt 2 (already evaluated).
[3] Skipping Prompt 3 (already evaluated).
[4] Skipping Prompt 4 (already evaluated).
[5] Skipping Prompt 5 (already evaluated).
[6] Skipping Prompt 6 (already evaluated).
[7] Skipping Prompt 7 (already evaluated).
[8] Skipping Prompt 8 (already evaluated).
[9] Skipping Prompt 9 (already evaluated).
[10] Skipping Prompt 10 (already evaluated).
[11] Skipping Prompt 11 (already evaluated).
[12] Skipping Prompt 12 (already evaluated).
[13] Skipping Prompt 13 (already evaluated).
[14] Skipping Prompt 14 (already evaluated).
[15] Skipping Prompt 15 (already evaluated).
[16] ⚠️ Format error (attempt 1) for Prompt 16:
I cannot provide scores because the necessary information is missing. The prompt doesn't contain the patient's age and education level needed to evaluate the message, and the response itself points out this inconsistency rather than providing a diagnosis message to evaluate.
[16] ⚠️ Format erro

KeyboardInterrupt: 